In [215]:
import os

from langchain_core.messages import HumanMessage
from langchain.tools import BaseTool, StructuredTool, tool
from langchain import hub

from langchain_ollama.chat_models import ChatOllama
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder


In [216]:
template="""
<|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023

{{ if .System }}{{ .System }}
{{- end }}
{{- if .Tools }}When you receive a tool call response, use the output to format an answer to the orginal user question.

{{- end }}<|eot_id|>
{{- range $i, $_ := .Messages }}
{{- $last := eq (len (slice $.Messages $i)) 1 }}
{{- if eq .Role "user" }}<|start_header_id|>user<|end_header_id|>
{{- if and $.Tools $last }}

Given the following functions, please respond with a JSON for a function call with its proper arguments that best answers the given prompt, or respond with a message if no function is needed.

If responding with a function call, use the format {"name": function name, "parameters": dictionary of argument name and its value}. Do not use variables.

{{ range $.Tools }}
{{- . }}
{{ end }}
{{ .Content }}<|eot_id|>
{{- else }}

{{ .Content }}<|eot_id|>
{{- end }}{{ if $last }}<|start_header_id|>assistant<|end_header_id|>

{{ end }}
{{- else if eq .Role "assistant" }}<|start_header_id|>assistant<|end_header_id|>
{{- if .ToolCalls }}
{{ range .ToolCalls }}
{"name": "{{ .Function.Name }}", "parameters": {{ .Function.Arguments }}}{{ end }}
{{- else }}

{{ .Content }}
{{- end }}{{ if not $last }}<|eot_id|>{{ end }}
{{- else if eq .Role "tool" }}<|start_header_id|>ipython<|end_header_id|>

{{ .Content }}<|eot_id|>{{ if $last }}<|start_header_id|>assistant<|end_header_id|>

{{ end }}
{{- end }}
{{- end }}
"""

In [244]:
n_gpu_layers = 1  # The number of layers to put on the GPU. The rest will be on the CPU. If you don't know how many layers there are, you can use -1 to move all to GPU.
n_batch = 512  # Should be between 1 and n_ctx, consider the amount of RAM of your Apple Silicon Chip.
# Make sure the model path is correct for your system!
llm = ChatOllama(
    model="llama3.2",
    n_batch=n_batch,
    f16_kv=True,  # MUST set to True, otherwise you will run into problem after a couple of calls
    verbose=True,  # Verbose is required to pass to the callback manager
    template=template,
)

In [235]:
@tool
def offer(item: str, price: float):
  """
  Seller offers an item for sale at a given price.
  """
  print(f"OFFERING {item} FOR {price}")
  pass

@tool
def sell(item: str, price: float):
  """
  Seller sells an item at a given price. This can only be done after the shopkeeper has accepted an offer
  """
  print(f"SELLING {item} FOR {price}")
  pass

@tool
def rescind_offer(item: str):
  """
  Rescind an offer for an item. This means the seller is no longer willing to sell the item.
  """
  print(f"RESCINDING OFFER FOR {item}")
  pass

@tool
def leave_shop():
  """
  Leave the shop. This means the seller is no longer interested in selling anything and ends the conversation.
  """
  print("LEAVING SHOP")
  pass

@tool
def state_name(name: str):
  """
  States your name
  """
  print("MY NAME IS", name)
  pass

In [255]:
# Initialize model
tools = [offer, sell, rescind_offer, leave_shop, state_name]
prompt_template = ChatPromptTemplate([
	("system", """
Your name is Erik Stoneforge. Erik Stoneforge is a seasoned adventurer in his mid-thirties, known for his sharp eye and shrewdness in negotiations. He steps into the pawn shop with a worn leather satchel containing carefully selected goods from his recent journey.
Inside are a finely crafted silver dagger etched with mysterious runes, which Erik believes holds more value to collectors of rare weaponry than to any common buyer. He also carries a cracked mana crystal, still faintly glowing, knowing it’s imperfect but hoping to fetch a decent price from someone seeking magical components. Lastly, an ancient bronze amulet adorned with emeralds catches the eye, and Erik is keen to emphasize the historical significance of the piece to drive up its value.
Erik prefers to haggle based on the uniqueness or rarity of each item, especially when he senses a merchant might undervalue magical or historical goods. He’s patient but firm in his negotiations, and while he’s willing to compromise on the mana crystal, he’s prepared to walk away if he doesn’t get a good offer for the amulet or the dagger.
	"""),
    MessagesPlaceholder("msgs")
])

model_with_tools = llm.bind_tools(tools)
query = "Hello! What's your name?"
messages = [HumanMessage(query)]
prompt = prompt_template.invoke({"msgs": messages})

ai_msg = model_with_tools.invoke(prompt)
messages.append(ai_msg)
messages

[chain/start] [prompt:ChatPromptTemplate] Entering Prompt run with input:
[inputs]
[chain/end] [prompt:ChatPromptTemplate] s] Exiting Prompt run with output:
[outputs]
[llm/start] [llm:ChatOllama] Entering LLM run with input:
{
  "prompts": [
    "System: \nYour name is Erik Stoneforge. Erik Stoneforge is a seasoned adventurer in his mid-thirties, known for his sharp eye and shrewdness in negotiations. He steps into the pawn shop with a worn leather satchel containing carefully selected goods from his recent journey.\nInside are a finely crafted silver dagger etched with mysterious runes, which Erik believes holds more value to collectors of rare weaponry than to any common buyer. He also carries a cracked mana crystal, still faintly glowing, knowing it’s imperfect but hoping to fetch a decent price from someone seeking magical components. Lastly, an ancient bronze amulet adorned with emeralds catches the eye, and Erik is keen to emphasize the historical significance of the piece to dr

[HumanMessage(content="Hello! What's your name?", additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2024-10-24T03:51:00.883774Z', 'message': {'role': 'assistant', 'content': '', 'tool_calls': [{'function': {'name': 'state_name', 'arguments': {'name': 'Erik Stoneforge'}}}]}, 'done_reason': 'stop', 'done': True, 'total_duration': 1556382958, 'load_duration': 30958125, 'prompt_eval_count': 620, 'prompt_eval_duration': 1212678000, 'eval_count': 16, 'eval_duration': 310249000}, id='run-e5152e35-44a3-471d-b291-fe28e1a868d9-0', tool_calls=[{'name': 'state_name', 'args': {'name': 'Erik Stoneforge'}, 'id': '8c11ba13-3885-4f5f-9840-91465aafee87', 'type': 'tool_call'}], usage_metadata={'input_tokens': 620, 'output_tokens': 16, 'total_tokens': 636})]

In [249]:
from langchain.globals import set_debug

set_debug(True)

In [307]:
# Invoke the ai model again
prompt = prompt_template.invoke({"msgs": messages})
ai_msg = model_with_tools.invoke(prompt)
messages.append(ai_msg)
messages

[chain/start] [prompt:ChatPromptTemplate] Entering Prompt run with input:
[inputs]
[chain/end] [prompt:ChatPromptTemplate] s] Exiting Prompt run with output:
[outputs]
[llm/start] [llm:ChatOllama] Entering LLM run with input:
{
  "prompts": [
    "System: \nYour name is Erik Stoneforge. Erik Stoneforge is a seasoned adventurer in his mid-thirties, known for his sharp eye and shrewdness in negotiations. He steps into the pawn shop with a worn leather satchel containing carefully selected goods from his recent journey.\nInside are a finely crafted silver dagger etched with mysterious runes, which Erik believes holds more value to collectors of rare weaponry than to any common buyer. He also carries a cracked mana crystal, still faintly glowing, knowing it’s imperfect but hoping to fetch a decent price from someone seeking magical components. Lastly, an ancient bronze amulet adorned with emeralds catches the eye, and Erik is keen to emphasize the historical significance of the piece to dr

[HumanMessage(content="Hello! What's your name?", additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2024-10-24T03:51:00.883774Z', 'message': {'role': 'assistant', 'content': '', 'tool_calls': [{'function': {'name': 'state_name', 'arguments': {'name': 'Erik Stoneforge'}}}]}, 'done_reason': 'stop', 'done': True, 'total_duration': 1556382958, 'load_duration': 30958125, 'prompt_eval_count': 620, 'prompt_eval_duration': 1212678000, 'eval_count': 16, 'eval_duration': 310249000}, id='run-e5152e35-44a3-471d-b291-fe28e1a868d9-0', tool_calls=[{'name': 'state_name', 'args': {'name': 'Erik Stoneforge'}, 'id': '8c11ba13-3885-4f5f-9840-91465aafee87', 'type': 'tool_call'}], usage_metadata={'input_tokens': 620, 'output_tokens': 16, 'total_tokens': 636}),
 ToolMessage(content='null', name='state_name', tool_call_id='8c11ba13-3885-4f5f-9840-91465aafee87'),
 AIMessage(content="I'm Erik Stoneforge, a seasoned ad

In [308]:
# Call tools if the ai message contains tool calls
for tool_call in ai_msg.tool_calls:
    if tool_call["name"].lower() not in [tool.name.lower() for tool in tools]:
      messages.pop()
      break
    selected_tool = {
      "offer": offer,
      "sell": sell,
      "rescind_offer": rescind_offer,
      "leave_shop": leave_shop,
      "state_name": state_name
    }[tool_call["name"].lower()]
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)

[tool/start] [tool:leave_shop] Entering Tool run with input:
"{}"
LEAVING SHOP
[tool/end] [tool:leave_shop] s] Exiting Tool run with output:
"content='null' name='leave_shop' tool_call_id='d5ef879a-d14a-4dd2-b311-e04acaafea92'"


In [301]:
# User query, then get a response from AI
query = "Ok, you may leave"
messages.append(HumanMessage(query))
prompt = prompt_template.invoke({"msgs": messages})
ai_msg = model_with_tools.invoke(prompt)
messages.append(ai_msg)
messages

[chain/start] [prompt:ChatPromptTemplate] Entering Prompt run with input:
[inputs]
[chain/end] [prompt:ChatPromptTemplate] s] Exiting Prompt run with output:
[outputs]
[llm/start] [llm:ChatOllama] Entering LLM run with input:
{
  "prompts": [
    "System: \nYour name is Erik Stoneforge. Erik Stoneforge is a seasoned adventurer in his mid-thirties, known for his sharp eye and shrewdness in negotiations. He steps into the pawn shop with a worn leather satchel containing carefully selected goods from his recent journey.\nInside are a finely crafted silver dagger etched with mysterious runes, which Erik believes holds more value to collectors of rare weaponry than to any common buyer. He also carries a cracked mana crystal, still faintly glowing, knowing it’s imperfect but hoping to fetch a decent price from someone seeking magical components. Lastly, an ancient bronze amulet adorned with emeralds catches the eye, and Erik is keen to emphasize the historical significance of the piece to dr

[HumanMessage(content="Hello! What's your name?", additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2024-10-24T03:51:00.883774Z', 'message': {'role': 'assistant', 'content': '', 'tool_calls': [{'function': {'name': 'state_name', 'arguments': {'name': 'Erik Stoneforge'}}}]}, 'done_reason': 'stop', 'done': True, 'total_duration': 1556382958, 'load_duration': 30958125, 'prompt_eval_count': 620, 'prompt_eval_duration': 1212678000, 'eval_count': 16, 'eval_duration': 310249000}, id='run-e5152e35-44a3-471d-b291-fe28e1a868d9-0', tool_calls=[{'name': 'state_name', 'args': {'name': 'Erik Stoneforge'}, 'id': '8c11ba13-3885-4f5f-9840-91465aafee87', 'type': 'tool_call'}], usage_metadata={'input_tokens': 620, 'output_tokens': 16, 'total_tokens': 636}),
 ToolMessage(content='null', name='state_name', tool_call_id='8c11ba13-3885-4f5f-9840-91465aafee87'),
 AIMessage(content="I'm Erik Stoneforge, a seasoned ad

In [306]:
messages.pop()
messages

[HumanMessage(content="Hello! What's your name?", additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2024-10-24T03:51:00.883774Z', 'message': {'role': 'assistant', 'content': '', 'tool_calls': [{'function': {'name': 'state_name', 'arguments': {'name': 'Erik Stoneforge'}}}]}, 'done_reason': 'stop', 'done': True, 'total_duration': 1556382958, 'load_duration': 30958125, 'prompt_eval_count': 620, 'prompt_eval_duration': 1212678000, 'eval_count': 16, 'eval_duration': 310249000}, id='run-e5152e35-44a3-471d-b291-fe28e1a868d9-0', tool_calls=[{'name': 'state_name', 'args': {'name': 'Erik Stoneforge'}, 'id': '8c11ba13-3885-4f5f-9840-91465aafee87', 'type': 'tool_call'}], usage_metadata={'input_tokens': 620, 'output_tokens': 16, 'total_tokens': 636}),
 ToolMessage(content='null', name='state_name', tool_call_id='8c11ba13-3885-4f5f-9840-91465aafee87'),
 AIMessage(content="I'm Erik Stoneforge, a seasoned ad

In [309]:
messages

[HumanMessage(content="Hello! What's your name?", additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2024-10-24T03:51:00.883774Z', 'message': {'role': 'assistant', 'content': '', 'tool_calls': [{'function': {'name': 'state_name', 'arguments': {'name': 'Erik Stoneforge'}}}]}, 'done_reason': 'stop', 'done': True, 'total_duration': 1556382958, 'load_duration': 30958125, 'prompt_eval_count': 620, 'prompt_eval_duration': 1212678000, 'eval_count': 16, 'eval_duration': 310249000}, id='run-e5152e35-44a3-471d-b291-fe28e1a868d9-0', tool_calls=[{'name': 'state_name', 'args': {'name': 'Erik Stoneforge'}, 'id': '8c11ba13-3885-4f5f-9840-91465aafee87', 'type': 'tool_call'}], usage_metadata={'input_tokens': 620, 'output_tokens': 16, 'total_tokens': 636}),
 ToolMessage(content='null', name='state_name', tool_call_id='8c11ba13-3885-4f5f-9840-91465aafee87'),
 AIMessage(content="I'm Erik Stoneforge, a seasoned ad